# 🏠 Домашнее задание — Урок 8: Регрессия на настоящих данных

**Медленная версия — каждая строка объяснена.**

На уроке мы предсказывали баллы по часам подготовки — данные придумали сами.
Сегодня работаем с **настоящим датасетом**: цены на жильё в Калифорнии.
Он встроен в `sklearn` и грузится одной строкой — регистрация и токены не нужны.

Задача та же, что на уроке: **признак → число**. Только данные живые.

## Шаг 1. Загружаем настоящий датасет

In [ ]:
# fetch_california_housing — встроенный в sklearn реальный датасет о жилье
# as_frame=True — получаем удобную таблицу (как Excel внутри Python)
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)  # скачиваем данные
df = data.frame                                 # берём таблицу
df.head()                                       # смотрим первые 5 строк

### Что в таблице?

Каждая строка — район Калифорнии. Нас интересуют две колонки:
- **MedInc** — средний доход жителей района
- **MedHouseVal** — медианная цена дома (в сотнях тысяч $) — это то, что предсказываем

Логичная гипотеза: чем богаче район → тем дороже дома. Проверим линией.

## Шаг 2. Выбираем ОДИН признак и цель

In [ ]:
# X — признак (из чего предсказываем). Двойные скобки — это таблица с одной колонкой
# y — цель (что предсказываем)
X = df[['MedInc']]        # 👈 МЕНЯЙ ТОЛЬКО ЭТО ИМЯ, чтобы попробовать другой признак
y = df['MedHouseVal']     # цена дома — не трогаем

print('Признак:', X.columns[0])
print('Строк в данных:', len(X))

## Шаг 3. Делим на train и test

In [ ]:
# train — на чём учимся, test — на чём проверяем (как "понял vs зазубрил")
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% отложим на проверку
    random_state=42      # фиксируем случайность, чтобы результат повторялся
)
print('Учимся на', len(X_train), 'районах, проверяем на', len(X_test))

## Шаг 4. Строим и обучаем регрессию

In [ ]:
# LinearRegression — та самая "линейка через точки" с урока
from sklearn.linear_model import LinearRegression

model = LinearRegression()       # создаём модель
model.fit(X_train, y_train)      # обучаем: модель находит лучшую линию

print('Наклон линии:', round(model.coef_[0], 3))
# Наклон показывает: на сколько растёт цена, когда доход увеличивается на 1

## Шаг 5. Считаем ошибку (MAE)

In [ ]:
# MAE — в среднем на сколько модель промахивается
from sklearn.metrics import mean_absolute_error

predictions = model.predict(X_test)              # предсказываем на test
mae = mean_absolute_error(y_test, predictions)   # сравниваем с правдой

print('MAE =', round(mae, 3))
print('В среднем модель ошибается на', round(mae, 3), '(в сотнях тысяч $)')

## Шаг 6. Рисуем линию поверх точек

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.scatter(X_test, y_test, alpha=0.2, label='реальные данные')   # точки
plt.plot(X_test, predictions, color='red', linewidth=2, label='линия модели')  # линия
plt.xlabel('Средний доход района (MedInc)')
plt.ylabel('Цена дома (MedHouseVal)')
plt.title('Регрессия: доход → цена дома')
plt.legend()
plt.show()

## Шаг 7. Делаем свой прогноз

In [ ]:
import pandas as pd

# Предскажем цену для района со средним доходом = 8
my_value = pd.DataFrame({'MedInc': [8]})
result = model.predict(my_value)
print('Прогноз цены для дохода 8:', round(result[0], 3))

# 👉 Попробуй свои числа: замени 8 на другое значение и запусти снова

---
## ✍️ Что сдать (ответь текстом в конце ноутбука)

1. Чему равен **наклон** линии? Объясни своими словами, что он значит.
2. Чему равен **MAE**? На сколько в среднем ошибается модель?
3. Предскажи цену для дохода **3, 6 и 10**. Числа растут — логично?
4. Замени признак `MedInc` на `AveRooms` (среднее число комнат) в Шаге 2
   и запусти заново. MAE стал больше или меньше? Какой признак предсказывает лучше?

## ⭐ Задание со звёздочкой (для тех, кому мало)

Зарегистрируйся на **Kaggle** и скачай датасет **Fish Market** или **Student Performance**.
Загрузи его через `pd.read_csv()`, выбери один числовой признак и построй регрессию
по тем же 7 шагам. Сравни MAE со своим california-результатом.